In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('../databases/war_games.db')
conn.execute('PRAGMA foreign_keys = ON')

def q(sql):
    return pd.read_sql_query(sql, conn)

## My queries

### Top 10 seasons by WAR

In [2]:
q("""
SELECT p.nameFirst || ' ' || p.nameLast AS name,
       b.yearID, b.teamID, b.HR, b.RBI, b.WAR
FROM Batting b
JOIN People p ON b.playerID = p.playerID
ORDER BY b.WAR DESC
LIMIT 10
""")

,name,yearID,teamID,HR,RBI,WAR
0,Barry Bonds,2001,SFN,73,137,11.9
1,Barry Bonds,2002,SFN,46,110,11.8
2,Aaron Judge,2024,NYA,58,144,10.9
3,Aaron Judge,2022,NYA,62,131,10.8
4,Mookie Betts,2018,BOS,32,80,10.7
5,Barry Bonds,2004,SFN,45,101,10.6
6,Mike Trout,2012,LAA,30,83,10.5
7,Alex Rodriguez,2000,SEA,41,132,10.4
8,Mike Trout,2016,LAA,29,100,10.4
9,Sammy Sosa,2001,CHN,64,160,10.3


### Top 10 40 HRs / 40 2B seasons (ranked by sum)

In [3]:
q("""
SELECT p.nameFirst || ' ' || p.nameLast AS name,
       b.yearID, b.teamID, b.HR, b."2B", b.HR + b."2B" AS total, b.WAR
FROM Batting b
JOIN People p ON b.playerID = p.playerID
WHERE b.HR >= 40 AND b."2B" >= 40
ORDER BY total DESC
LIMIT 10
""")

,name,yearID,teamID,HR,2B,total,WAR
0,Todd Helton,2001,COL,49,54,103,7.8
1,Todd Helton,2000,COL,42,59,101,8.9
2,Carlos Delgado,2000,TOR,41,57,98,7.3
3,Albert Pujols,2004,SLN,46,51,97,8.5
4,Derrek Lee,2005,CHN,46,50,96,7.7
5,Chris Davis,2013,BAL,53,42,95,7.1
6,Albert Pujols,2003,SLN,43,51,94,8.7
7,Albert Pujols,2009,SLN,47,45,92,9.7
8,David Ortiz,2004,BOS,41,47,88,4.3
9,David Ortiz,2005,BOS,47,40,87,5.2


### Highest WAR between 2005 and 2009

In [4]:
q("""
SELECT p.nameFirst || ' ' || p.nameLast AS name,
       SUM(b.WAR) AS total_WAR,
       COUNT(*) AS seasons,
       SUM(b.HR) AS HR,
       ROUND(AVG(b.WAR), 1) AS avg_WAR
FROM Batting b
JOIN People p ON b.playerID = p.playerID
WHERE b.yearID BETWEEN 2005 AND 2009
GROUP BY b.playerID
ORDER BY total_WAR DESC
LIMIT 5
""")

,name,total_WAR,seasons,HR,avg_WAR
0,Albert Pujols,44.5,5,206,8.9
1,Chase Utley,39.6,5,146,7.9
2,Alex Rodriguez,34.3,5,202,6.9
3,Mark Teixeira,29.3,7,178,4.2
4,David Wright,27.3,5,126,5.5


### Players who appeared with NYY and ARI between 2001 and 2010

In [5]:
q("""
SELECT p.nameFirst || ' ' || p.nameLast AS name,
       SUM(CASE WHEN b.teamID = 'NYA' THEN b.WAR ELSE 0 END) AS WAR_NYA,
       SUM(CASE WHEN b.teamID = 'ARI' THEN b.WAR ELSE 0 END) AS WAR_ARI,
       SUM(CASE WHEN b.teamID IN ('NYA','ARI') THEN b.WAR ELSE 0 END) AS total_WAR
FROM Batting b
JOIN People p ON b.playerID = p.playerID
WHERE b.yearID BETWEEN 2001 AND 2010
AND b.playerID IN (
    SELECT playerID FROM Batting
    WHERE teamID = 'NYA' AND yearID BETWEEN 2001 AND 2010
)
AND b.playerID IN (
    SELECT playerID FROM Batting
    WHERE teamID = 'ARI' AND yearID BETWEEN 2001 AND 2010
)
GROUP BY b.playerID
ORDER BY total_WAR DESC
LIMIT 5
""")

,name,WAR_NYA,WAR_ARI,total_WAR
0,Raul Mondesi,2.1,0.5,2.6
1,Tony Clark,0.2,1.7,1.9
2,Chad Moeller,-0.1,1.4,1.3
3,Javier Vazquez,0.2,0.4,0.6
4,Richie Sexson,-0.2,0.3,0.1


### Franchises between 2000 and 2025

In [6]:
q("""
SELECT DISTINCT t.franchID, t.name
FROM (
    SELECT teamID, yearID FROM Batting
    UNION
    SELECT teamID, yearID FROM Pitching
) s
JOIN Teams t ON s.teamID = t.teamID AND s.yearID = t.yearID
GROUP BY t.franchID
ORDER BY franchID
""")

,franchID,name
0,ANA,Anaheim Angels
1,ARI,Arizona Diamondbacks
2,ATL,Atlanta Braves
3,BAL,Baltimore Orioles
4,BOS,Boston Red Sox
5,CHC,Chicago Cubs
6,CHW,Chicago White Sox
7,CIN,Cincinnati Reds
8,CLE,Cleveland Indians
9,COL,Colorado Rockies


## Intermediate tables

Sanity check: Every season in appearances corresponds to a season in batting or pitching

In [7]:
q("""
SELECT a.playerID, a.yearID, a.teamID
FROM Appearances a
LEFT JOIN Batting b ON a.playerID = b.playerID AND a.yearID = b.yearID AND a.teamID = b.teamID
LEFT JOIN Pitching p ON a.playerID = p.playerID AND a.yearID = p.yearID AND a.teamID = p.teamID
WHERE b.playerID IS NULL AND p.playerID IS NULL
""")

,playerID,yearID,teamID


In [8]:
with pd.option_context('display.max_rows', None):
    display(q("""
    SELECT DISTINCT awardID
    FROM AwardsPlayers
    ORDER BY awardID
    """))

,awardID
0,ALCS MVP
1,Cy Young Award
2,Gold Glove
3,Most Valuable Player
4,NLCS MVP
5,Silver Slugger
6,World Series MVP


In [9]:
queries = [
    ("Multiple MVPs", """
        SELECT p.nameFirst || ' ' || p.nameLast AS name, COUNT(*) AS mvps
        FROM AwardsPlayers a
        JOIN People p ON a.playerID = p.playerID
        WHERE a.awardID = 'Most Valuable Player'
        GROUP BY a.playerID
        HAVING COUNT(*) > 1
        ORDER BY mvps DESC
    """),
    ("Multiple Cy Youngs", """
        SELECT p.nameFirst || ' ' || p.nameLast AS name, COUNT(*) AS cy_youngs
        FROM AwardsPlayers a
        JOIN People p ON a.playerID = p.playerID
        WHERE a.awardID = 'Cy Young Award'
        GROUP BY a.playerID
        HAVING COUNT(*) > 1
        ORDER BY cy_youngs DESC
    """),
    ("MVP and Cy Young", """
        SELECT DISTINCT p.nameFirst || ' ' || p.nameLast AS name
        FROM AwardsPlayers a1
        JOIN AwardsPlayers a2 ON a1.playerID = a2.playerID
        JOIN People p ON a1.playerID = p.playerID
        WHERE a1.awardID = 'Most Valuable Player'
        AND a2.awardID = 'Cy Young Award'
    """),
    ("NLCS MVP and World Series MVP (same year)", """
        SELECT p.nameFirst || ' ' || p.nameLast AS name, a1.yearID
        FROM AwardsPlayers a1
        JOIN AwardsPlayers a2 ON a1.playerID = a2.playerID AND a1.yearID = a2.yearID
        JOIN People p ON a1.playerID = p.playerID
        WHERE a1.awardID = 'NLCS MVP'
        AND a2.awardID = 'World Series MVP'
        ORDER BY a1.yearID
    """),
]

for title, sql in queries:
    print(f'\n--- {title} ---')
    display(q(sql))



--- Multiple MVPs ---


,name,mvps
0,Barry Bonds,7
1,Shohei Ohtani,4
2,Mike Trout,3
3,Mike Schmidt,3
4,Alex Rodriguez,3
5,Albert Pujols,3
6,Stan Musial,3
7,Mickey Mantle,3
8,Aaron Judge,3
9,Jimmie Foxx,3



--- Multiple Cy Youngs ---


,name,cy_youngs
0,Roger Clemens,7
1,Randy Johnson,5
2,Greg Maddux,4
3,Steve Carlton,4
4,Justin Verlander,3
5,Tom Seaver,3
6,Max Scherzer,3
7,Jim Palmer,3
8,Pedro Martinez,3
9,Sandy Koufax,3



--- MVP and Cy Young ---


,name
0,Vida Blue
1,Roger Clemens
2,Dennis Eckersley
3,Rollie Fingers
4,Bob Gibson
5,Willie Hernandez
6,Clayton Kershaw
7,Sandy Koufax
8,Denny McLain
9,Don Newcombe



--- NLCS MVP and World Series MVP (same year) ---


,name,yearID
0,Willie Stargell,1979
1,Darrell Porter,1982
2,Orel Hershiser,1988
3,Livan Hernandez,1997
4,Cole Hamels,2008
5,David Freese,2011
6,Madison Bumgarner,2014
7,Corey Seager,2020


In [10]:
q("""
SELECT p.nameFirst || ' ' || p.nameLast AS name, COUNT(*) AS ws_mvps,
       GROUP_CONCAT(a.yearID) AS years
FROM AwardsPlayers a
JOIN People p ON a.playerID = p.playerID
WHERE a.awardID = 'World Series MVP'
GROUP BY a.playerID
HAVING COUNT(*) > 1
ORDER BY ws_mvps DESC
""")


,name,ws_mvps,years
0,Corey Seager,2,"2020,2023"
1,Sandy Koufax,2,"1963,1965"
2,Reggie Jackson,2,"1973,1977"
3,Bob Gibson,2,"1964,1967"


### Hall of Fame

In [11]:
q("""
SELECT DISTINCT p.nameFirst || ' ' || p.nameLast AS name, h.yearid AS inducted_year, h.votedBy
FROM HallOfFame h
JOIN People p ON h.playerID = p.playerID
WHERE h.playerID IN (
    SELECT playerID FROM Batting
    UNION
    SELECT playerID FROM Pitching
)
ORDER BY h.yearid DESC
""")


,name,inducted_year,votedBy
0,Jeff Kent,2026,Contemporary Baseball Era
1,CC Sabathia,2025,BBWAA
2,Ichiro Suzuki,2025,BBWAA
3,Billy Wagner,2025,BBWAA
4,Adrian Beltre,2024,BBWAA
5,Todd Helton,2024,BBWAA
6,Joe Mauer,2024,BBWAA
7,Fred McGriff,2023,Contemporary Era
8,Scott Rolen,2023,BBWAA
9,David Ortiz,2022,BBWAA


In [32]:
q("""
SELECT DISTINCT p.nameFirst || ' ' || p.nameLast AS name, h.yearid AS inducted_year
FROM HallOfFame h
JOIN People p ON h.playerID = p.playerID
WHERE h.playerID IN (
    SELECT playerID FROM Batting
    UNION
    SELECT playerID FROM Pitching
)
AND h.playerID NOT IN (
    SELECT playerID FROM AwardsPlayers
    WHERE awardID IN ('Most Valuable Player', 'Cy Young Award')
)
ORDER BY h.yearid DESC
""")

,name,inducted_year
0,Billy Wagner,2025
1,Adrian Beltre,2024
2,Todd Helton,2024
3,Fred McGriff,2023
4,Scott Rolen,2023
5,David Ortiz,2022
6,Derek Jeter,2020
7,Harold Baines,2019
8,Edgar Martinez,2019
9,Mike Mussina,2019


In [12]:
q("""
SELECT DISTINCT p.nameFirst || ' ' || p.nameLast AS name, h.yearid AS inducted_year,
       GROUP_CONCAT(DISTINCT a.awardID) AS awards
FROM HallOfFame h
JOIN People p ON h.playerID = p.playerID
JOIN AwardsPlayers a ON h.playerID = a.playerID
    AND a.awardID IN ('Most Valuable Player', 'Cy Young Award')
WHERE h.playerID IN (
    SELECT playerID FROM Batting
    UNION
    SELECT playerID FROM Pitching
)
GROUP BY h.playerID
ORDER BY h.yearid DESC
""")

,name,inducted_year,awards
0,Jeff Kent,2026,Most Valuable Player
1,CC Sabathia,2025,Cy Young Award
2,Ichiro Suzuki,2025,Most Valuable Player
3,Joe Mauer,2024,Most Valuable Player
4,Larry Walker,2020,Most Valuable Player
5,Roy Halladay,2019,Cy Young Award
6,Vladimir Guerrero,2018,Most Valuable Player
7,Chipper Jones,2018,Most Valuable Player
8,Jeff Bagwell,2017,Most Valuable Player
9,Ivan Rodriguez,2017,Most Valuable Player


In [13]:
print('--- Hall of Fame Batters ---')
display(q("""
SELECT p.nameFirst || ' ' || p.nameLast AS name,
       h.yearid AS inducted_year,
       SUM(b.WAR) AS total_WAR,
       SUM(b.HR) AS HR,
       COUNT(*) AS seasons
FROM HallOfFame h
JOIN People p ON h.playerID = p.playerID
JOIN Batting b ON h.playerID = b.playerID
GROUP BY h.playerID
ORDER BY total_WAR DESC
"""))

print('--- Hall of Fame Pitchers ---')
display(q("""
SELECT p.nameFirst || ' ' || p.nameLast AS name,
       h.yearid AS inducted_year,
       SUM(pw.WAR) AS total_WAR,
       SUM(pw.W) AS W,
       COUNT(*) AS seasons
FROM HallOfFame h
JOIN People p ON h.playerID = p.playerID
JOIN Pitching pw ON h.playerID = pw.playerID
GROUP BY h.playerID
ORDER BY total_WAR DESC
"""))


--- Hall of Fame Batters ---


,name,inducted_year,total_WAR,HR,seasons
0,Adrian Beltre,2024,89.8,455,19
1,Ichiro Suzuki,2025,60.0,117,20
2,Chipper Jones,2018,58.4,315,13
3,Todd Helton,2024,55.9,304,14
4,Joe Mauer,2024,55.6,143,15
5,David Ortiz,2022,55.0,531,17
6,Scott Rolen,2023,54.6,234,15
7,Derek Jeter,2020,48.0,197,15
8,Vladimir Guerrero,2018,46.0,357,12
9,Jim Thome,2018,43.5,416,16


--- Hall of Fame Pitchers ---


,name,inducted_year,total_WAR,W,seasons
0,Roy Halladay,2019,62.4,194,14
1,CC Sabathia,2025,61.9,251,20
2,Randy Johnson,2015,51.3,143,10
3,Pedro Martinez,2015,45.7,112,10
4,Mariano Rivera,2019,41.3,56,14
5,Mike Mussina,2019,40.8,134,9
6,Greg Maddux,2014,29.7,134,11
7,Tom Glavine,2014,27.7,118,9
8,John Smoltz,2015,23.3,56,10
9,Billy Wagner,2025,20.1,30,12


### Appearances

In [14]:
q("""
SELECT franchID, name, games_at_2B FROM (
    SELECT t.franchID,
           p.nameFirst || ' ' || p.nameLast AS name,
           SUM(a.G_2b) AS games_at_2B,
           RANK() OVER (PARTITION BY t.franchID ORDER BY SUM(a.G_2b) DESC) AS rnk
    FROM Appearances a
    JOIN People p ON a.playerID = p.playerID
    JOIN Teams t ON a.teamID = t.teamID AND a.yearID = t.yearID
    GROUP BY t.franchID, a.playerID
    HAVING games_at_2B > 0
)
WHERE rnk = 1
ORDER BY games_at_2B DESC
LIMIT 30
""")



,franchID,name,games_at_2B
0,HOU,Jose Altuve,1831
1,CIN,Brandon Phillips,1586
2,BOS,Dustin Pedroia,1492
3,PHI,Chase Utley,1453
4,NYY,Robinson Cano,1350
5,BAL,Brian Roberts,1213
6,CLE,Jason Kipnis,1050
7,MIL,Rickie Weeks,1044
8,TEX,Ian Kinsler,1029
9,ATL,Ozzie Albies,1025


## Towards an Immaculate Grid

In [15]:
from itertools import product

# --- Condition resolvers ---
# Each returns a set of playerIDs matching the condition.
# To add new conditions, just add a new resolver function and register it.

def _resolve_franchise(label):
    """Players who played for a franchise (e.g. 'NYY', 'LAD')."""
    df = q(f"""
        SELECT DISTINCT b.playerID
        FROM Batting b
        JOIN Teams t ON b.teamID = t.teamID AND b.yearID = t.yearID
        WHERE t.franchID = '{label}'
    """)
    return set(df['playerID'])

def _resolve_award(label):
    """Players who won an award (e.g. 'Gold Glove', 'MVP')."""
    award_map = {
        'MVP': 'Most Valuable Player',
        'Cy Young': 'Cy Young Award',
        'Gold Glove': 'Gold Glove',
        'Silver Slugger': 'Silver Slugger',
        'WS MVP': 'World Series MVP',
        'ALCS MVP': 'ALCS MVP',
        'NLCS MVP': 'NLCS MVP',
    }
    award_name = award_map.get(label, label)
    df = q(f"""
        SELECT DISTINCT playerID
        FROM AwardsPlayers
        WHERE awardID = '{award_name}'
    """)
    return set(df['playerID'])

def _resolve_hof(_label):
    """Players inducted into the Hall of Fame."""
    df = q("SELECT DISTINCT playerID FROM HallOfFame")
    return set(df['playerID'])

# Registry: prefix -> resolver
_RESOLVERS = {
    'award': _resolve_award,
    'hof': _resolve_hof,
}

# All known franchise codes
_FRANCHISE_CODES = {
    'ANA', 'ARI', 'ATL', 'BAL', 'BOS', 'CHC', 'CHW', 'CIN', 'CLE', 'COL',
    'DET', 'FLA', 'HOU', 'KCR', 'LAD', 'MIL', 'MIN', 'NYM', 'NYY', 'OAK',
    'PHI', 'PIT', 'SDP', 'SEA', 'SFG', 'STL', 'TBD', 'TEX', 'TOR', 'WSN',
}

def resolve_condition(label):
    """
    Resolve a condition label to a set of playerIDs.

    Formats:
        'NYY'           -> franchise
        'Gold Glove'    -> award (matched by name)
        'MVP'           -> award (shorthand)
        'Cy Young'      -> award (shorthand)
        'HoF'           -> Hall of Fame
    """
    if label.upper() in _FRANCHISE_CODES:
        return _resolve_franchise(label.upper())
    if label in ('HoF', 'Hall of Fame'):
        return _resolve_hof(label)
    # Try as award
    return _resolve_award(label)


def _get_war_lookup(player_ids):
    """Build name + career WAR lookup for a set of playerIDs."""
    if not player_ids:
        return {}
    placeholders = ','.join(f"'{pid}'" for pid in player_ids)
    df = q(f"""
        SELECT b.playerID, p.nameFirst || ' ' || p.nameLast AS name,
               SUM(b.WAR) AS career_WAR
        FROM Batting b
        JOIN People p ON b.playerID = p.playerID
        WHERE b.playerID IN ({placeholders})
        GROUP BY b.playerID
    """)
    return {row['playerID']: (row['name'], row['career_WAR']) for _, row in df.iterrows()}


def immaculate_grid(conditions):
    """
    Takes a list of 6 conditions. First 3 are rows, last 3 are columns.
    Each condition can be a franchise code, award name, or 'HoF'.

    Returns a 3x3 grid of players satisfying both row and column conditions.
    No player repeated. Prefers highest career WAR.
    """
    rows = conditions[:3]
    cols = conditions[3:]

    # Resolve all conditions to player sets
    resolved = {}
    for label in set(rows + cols):
        resolved[label] = resolve_condition(label)

    # For each cell, find candidates (intersection of row and col sets)
    cells = list(product(rows, cols))
    all_candidates = set()
    candidates = {}
    for r, c in cells:
        both = resolved[r] & resolved[c]
        candidates[(r, c)] = both
        all_candidates |= both

    # Build WAR lookup for all candidate players
    lookup = _get_war_lookup(all_candidates)

    # Sort each cell's candidates by WAR descending
    for cell in candidates:
        candidates[cell] = sorted(
            [pid for pid in candidates[cell] if pid in lookup],
            key=lambda pid: lookup[pid][1],
            reverse=True,
        )

    # Backtracking solver: most-constrained-first
    cell_order = sorted(cells, key=lambda cell: len(candidates[cell]))
    used = set()
    grid = {}

    def solve(idx):
        if idx == len(cell_order):
            return True
        cell = cell_order[idx]
        for pid in candidates[cell]:
            if pid not in used:
                used.add(pid)
                grid[cell] = pid
                if solve(idx + 1):
                    return True
                used.remove(pid)
                del grid[cell]
        return False

    if not solve(0):
        print("No valid grid found!")
        return None

    # Display
    header = [''] + cols
    table_rows = []
    for r in rows:
        row_data = [r]
        for c in cols:
            pid = grid[(r, c)]
            name, war = lookup[pid]
            row_data.append(f'{name} ({war:.1f})')
        table_rows.append(row_data)

    result = pd.DataFrame(table_rows, columns=header)
    display(result)
    return result

In [16]:
# Franchises only (same as before)
immaculate_grid(['NYY', 'BOS', 'LAD', 'CHC', 'ATL', 'HOU']);

,,CHC,ATL,HOU
0,NYY,Anthony Rizzo (40.3),Robinson Cano (68.7),Carlos Beltran (65.3)
1,BOS,Justin Turner (38.5),J. D. Drew (41.4),Alex Bregman (43.1)
2,LAD,Cody Bellinger (30.5),Freddie Freeman (64.2),Jason Heyward (41.5)


In [17]:
# Mix franchises and awards
immaculate_grid(['NYY', 'BOS', 'LAD', 'MVP', 'Gold Glove', 'HoF']);

,,MVP,Gold Glove,HoF
0,NYY,Alex Rodriguez (89.8),Robinson Cano (68.7),Ichiro Suzuki (60.0)
1,BOS,Mookie Betts (75.1),Ian Kinsler (53.9),Adrian Beltre (89.8)
2,LAD,Albert Pujols (101.3),Freddie Freeman (64.2),Jim Thome (43.5)


In [18]:
# Awards vs franchises
immaculate_grid(['MVP', 'Cy Young', 'Silver Slugger', 'ATL', 'SFG', 'HOU']);

,,ATL,SFG,HOU
0,MVP,Freddie Freeman (64.2),Barry Bonds (59.2),Jose Altuve (53.2)
1,Cy Young,Tom Glavine (1.6),Jake Peavy (2.0),Zack Greinke (5.2)
2,Silver Slugger,Robinson Cano (68.7),Carlos Beltran (65.3),Miguel Tejada (44.2)


In [19]:
# AL vs NL rivals
immaculate_grid(['NYY', 'BOS', 'CHW', 'NYM', 'ATL', 'CHC']);

,,NYM,ATL,CHC
0,NYY,Carlos Beltran (65.3),Robinson Cano (68.7),Anthony Rizzo (40.3)
1,BOS,Adrian Gonzalez (43.8),J. D. Drew (41.4),Justin Turner (38.5)
2,CHW,Todd Frazier (25.9),Andruw Jones (44.7),Austin Jackson (22.0)


In [20]:
# HoF and awards vs NL East franchises
immaculate_grid(['HoF', 'MVP', 'Gold Glove', 'NYM', 'PHI', 'ATL']);

,,NYM,PHI,ATL
0,HoF,Mike Piazza (18.1),Scott Rolen (54.6),Chipper Jones (58.4)
1,MVP,Rickey Henderson (1.8),Bryce Harper (53.7),Freddie Freeman (64.2)
2,Gold Glove,Carlos Beltran (65.3),Andrew McCutchen (49.1),Robinson Cano (68.7)
